# Day 8 · PyTorch Data Pipelines
Custom `Dataset` for the tabular farm data, `DataLoader` with batch size 32, per-batch status.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv("crop_yield.csv")          # put crop_yield.csv next to this notebook
print(df.shape)
df.head()

(1000000, 10)


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443
3,North,Sandy,Soybean,986.866331,16.644190,False,True,Rainy,146,6.517573
4,South,Silt,Wheat,730.379174,31.620687,True,True,Cloudy,110,7.248251


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader

class FarmDataset(Dataset):
    """Tabular farm data -> (features, yield) tensors."""
    def __init__(self, frame, target="Yield_tons_per_hectare"):
        feats = pd.get_dummies(frame.drop(columns=[target]), columns=["Region", "Soil_Type", "Crop", "Weather_Condition"]).astype(float)
        self.columns = list(feats.columns)
        x = feats.to_numpy(dtype=np.float32); self.mean, self.std = x.mean(0), x.std(0) + 1e-8
        self.x = torch.from_numpy((x - self.mean) / self.std)
        self.y = torch.from_numpy(frame[target].to_numpy(dtype=np.float32))
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i]

ds = FarmDataset(df)
print("samples:", len(ds), "| features:", len(ds.columns))
loader = DataLoader(ds, batch_size=32, shuffle=True)
print("batches per epoch:", len(loader))

samples: 1000000 | features: 24
batches per epoch: 31250


In [3]:
# Per-batch status: first 5 batches, then every 5000th (printing all 31k lines would drown the notebook)
running = 0.0
for i, (xb, yb) in enumerate(loader):
    running += yb.mean().item()
    if i < 5 or i % 5000 == 0 or i == len(loader) - 1:
        print(f"batch {i:5d}/{len(loader)}  x{tuple(xb.shape)}  y{tuple(yb.shape)}  mean yield {yb.mean():.3f}  nan-free: {not torch.isnan(xb).any().item()}")
print("average of batch means:", round(running / len(loader), 4), "| dataset mean:", round(ds.y.mean().item(), 4))

batch     0/31250  x(32, 24)  y(32,)  mean yield 4.857  nan-free: True
batch     1/31250  x(32, 24)  y(32,)  mean yield 5.102  nan-free: True
batch     2/31250  x(32, 24)  y(32,)  mean yield 4.391  nan-free: True
batch     3/31250  x(32, 24)  y(32,)  mean yield 4.854  nan-free: True
batch     4/31250  x(32, 24)  y(32,)  mean yield 5.108  nan-free: True


batch  5000/31250  x(32, 24)  y(32,)  mean yield 4.483  nan-free: True


batch 10000/31250  x(32, 24)  y(32,)  mean yield 4.778  nan-free: True


batch 15000/31250  x(32, 24)  y(32,)  mean yield 4.426  nan-free: True


batch 20000/31250  x(32, 24)  y(32,)  mean yield 4.294  nan-free: True


batch 25000/31250  x(32, 24)  y(32,)  mean yield 5.018  nan-free: True


batch 30000/31250  x(32, 24)  y(32,)  mean yield 4.763  nan-free: True
batch 31249/31250  x(32, 24)  y(32,)  mean yield 4.780  nan-free: True


average of batch means: 4.6495 | dataset mean: 4.6495
